# Task 2: run on a GPU

Task 1 is complete. The handwritten Task 2 normalizer is ready; this notebook generates and measures its audio with the original pinned baseline. It does not run Task 3.

Run the cells in order. Use a GPU runtime. Keep the original organizer kit and your updated `code/`, `predictions/`, `results.json`, and `idea.md` together in one folder. Copy `reports/task2/` too when using the working project; when using the new ZIP or repository, keep its `code/task2_evidence/` folder instead. The runner can restore the frozen-rules evidence from that packaged folder. The public submission alone does not include the kit data or baseline code.

For Colab, copy that folder to Google Drive, for example `MyDrive/Inter-IIT TTS`. For Kaggle, enable GPU and internet access, copy the kit to a writable folder such as `/kaggle/working/tts-kit`, and change `KIT` below. Keep model downloads and the organizer's data out of the public repository.

If you already submitted the midterm form, leave parts (1)–(3) of `idea.md` unchanged.

## 1. Find your kit folder

Colab will ask permission to mount your Google Drive. Check `KIT` before running the next cell. A Kaggle input folder is read-only, so use a copied folder under `/kaggle/working`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')

# Change this if your folder has a different name or location.
KIT = Path('/content/drive/MyDrive/Inter-IIT TTS' if IN_COLAB else '/kaggle/working/tts-kit')
KIT = KIT.expanduser().resolve()
required = [
    'baseline/requirements.txt', 'baseline/config.yaml', 'baseline/run_baseline.py',
    'metrics.py', 'check_format.py', 'data/val_items.csv', 'data/eval_items.csv',
    'data/prompts.csv', 'data/prompts', 'data/baseline_val',
    'code/run_task2.py', 'code/record_task2_results.py', 'code/normalize.py',
    'code/build_midterm.py', 'predictions/t2_normalized.csv',
    'results_template.json', 'results.json', 'idea.md',
]
missing = [name for name in required if not (KIT / name).exists()]
frozen_evidence = [KIT / 'reports/task2/rules_frozen.json', KIT / 'code/task2_evidence/rules_frozen.json']
if not any(path.exists() for path in frozen_evidence):
    missing.append('reports/task2/rules_frozen.json or code/task2_evidence/rules_frozen.json')
if missing:
    raise FileNotFoundError('Your KIT folder is incomplete. Copy the original kit and updated files. Missing: ' + ', '.join(missing))
os.chdir(KIT)
print('Kit found. Python:', sys.version.split()[0])


## 2. Install the organizer's pinned dependencies

This uses the original `baseline/requirements.txt` without changing its versions. Installation can take several minutes.

If this changes NumPy and you see a NumPy import error later, restart the runtime once, then run from cell 1 again. Your Drive files remain saved.

In [ ]:
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-r', str(KIT / 'baseline/requirements.txt')
])


## 3. Check the GPU and Hugging Face model access

Accept the IndicF5 access conditions on Hugging Face if the model page asks you to. A read-access token can be supplied through the Colab secret `HF_TOKEN`, an existing `HF_TOKEN` environment variable, or the hidden prompt below. Never paste it into a notebook cell or commit it.

This cell checks access to the pinned model configuration; the runner downloads the model weights when synthesis starts.

In [ ]:
import getpass
import torch
import yaml
from huggingface_hub import get_token, hf_hub_download

if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is available. Select a GPU runtime, then rerun this notebook.')
print('GPU:', torch.cuda.get_device_name(0))

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

hf_token = os.environ.get('HF_TOKEN')
if not hf_token and IN_COLAB:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
    except Exception:
        hf_token = None
if not hf_token:
    hf_token = get_token()
if not hf_token:
    hf_token = getpass.getpass('Hugging Face read token (hidden): ').strip()
if not hf_token:
    raise RuntimeError('A Hugging Face token is needed to check model access.')

# Pass the token to child processes in memory, without writing it to this notebook.
os.environ['HF_TOKEN'] = hf_token
settings = yaml.safe_load((KIT / 'baseline/config.yaml').read_text(encoding='utf-8'))
try:
    hf_hub_download(
        repo_id=settings['repo'], revision=settings['revision'],
        filename='config.json', token=hf_token,
    )
except Exception:
    raise RuntimeError(
        'Model access could not be verified. Check your internet connection, '
        'Hugging Face token permissions, and acceptance of the IndicF5 access conditions.'
    ) from None
finally:
    del hf_token
print('Pinned IndicF5 model access verified.')


## 4. Generate and measure validation audio

This stage scores the supplied baseline recordings, generates normalization-only audio for all **24 validation items** using the original model settings and matching seeds, and measures CER and speaker similarity. It saves that audio in `predictions/val_control/` and records the measured Task 2 results.

Wait for the cell to finish. If the validation failure guard stops it, inspect the saved scores and fix the reported problem before generating evaluation audio. A failed run must not be treated as a completed experiment.

In [ ]:
def run_task2(stage):
    subprocess.check_call([
        sys.executable, '-u', str(KIT / 'code/run_task2.py'), '--stage', stage
    ], cwd=str(KIT))

run_task2('val')


## 5. Read the validation results

CER measures character errors: lower is better. SIM measures how closely the generated voice matches the assigned reference speaker: higher is better. These are validation measurements, not scores on the hidden final evaluation.

In [ ]:
import json
from IPython.display import JSON, display

for name in ('baseline_scores.json', 'val_scores.json', 'comparison.json'):
    score_path = KIT / 'reports/task2' / name
    if score_path.exists():
        print(name)
        display(JSON(json.loads(score_path.read_text(encoding='utf-8'))))
    else:
        print(name + ': not saved yet; check the validation runner output.')


## 6. Listen to two validation examples

Compare the supplied baseline with the normalization-only result. Listen for the intended numbers and abbreviations, and check whether the voice still resembles the reference. These recordings are for human validation; no evaluation recordings are played here.

In [ ]:
from IPython.display import Audio

item_id = 'val_hd_01'
for label, folder in [('Supplied baseline', 'data/baseline_val'), ('Task 2 normalization', 'predictions/val_control')]:
    wav = KIT / folder / (item_id + '.wav')
    if not wav.exists():
        raise FileNotFoundError('Run the validation stage first: ' + str(wav.relative_to(KIT)))
    print(item_id + ' — ' + label)
    display(Audio(filename=str(wav)))


In [ ]:
item_id = 'val_hd_08'
for label, folder in [('Supplied baseline', 'data/baseline_val'), ('Task 2 normalization', 'predictions/val_control')]:
    wav = KIT / folder / (item_id + '.wav')
    if not wav.exists():
        raise FileNotFoundError('Run the validation stage first: ' + str(wav.relative_to(KIT)))
    print(item_id + ' — ' + label)
    display(Audio(filename=str(wav)))


## 7. Generate the Task 2 evaluation submission

Run this only after the validation stage passes. It uses the already frozen `predictions/t2_normalized.csv` and unchanged baseline settings to generate audio for the **40 evaluation items**, saving it in `predictions/t2_norm/`. It checks the frozen source evidence and does not rewrite the normalized CSV.

Do not listen to evaluation audio or change rules based on evaluation items. The evaluation stage performs structural checks and prints progress without displaying input text.

In [ ]:
run_task2('eval')


## 8. Build and check the ZIP

This runs the submission builder and `check_format.py`. Fix any `FAIL` before submitting. `MISSING` identifies work that has not yet been completed; the final submission must meet the final requirements.

In Colab the next cell downloads your ZIP. In Kaggle, find it in the `KIT` folder and download it through the notebook's file browser. Notebook outputs are cleared in the packaged copy, so embedded organizer audio and tokens are not published.

In [ ]:
run_task2('package')

roll_no = json.loads((KIT / 'results.json').read_text(encoding='utf-8'))['roll_no']
archive = KIT / (roll_no + '_tts_bootcamp.zip')
if not archive.exists():
    raise FileNotFoundError('The package stage did not create the expected ZIP.')
print('Submission ZIP:', archive.name)
if IN_COLAB:
    from google.colab import files
    files.download(str(archive))
else:
    print('Download the ZIP from your KIT folder in the file browser.')
